In [3]:
#Cat vs Dog Dataset CNN

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers,models,regularizers,optimizers,callbacks,Sequential
from tensorflow.keras.preprocessing import image
from sklearn.metrics import classification_report, confusion_matrix

import kagglehub
import os

In [5]:
dataset_path = kagglehub.dataset_download("vrajesh0sharma7/cat-vs-dog-classification")

100%|██████████| 545M/545M [00:13<00:00, 41.3MB/s]

Extracting files...


In [6]:
print (f"Dataset Path: {dataset_path}")

Dataset Path: /root/.cache/kagglehub/datasets/vrajesh0sharma7/cat-vs-dog-classification/versions/1


In [7]:
train_dir = os.path.join(dataset_path,'dogs_vs_cats','train')
val_dir = os.path.join(dataset_path,'dogs_vs_cats','test')

In [8]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(150,150),
    batch_size=32,
    label_mode = 'categorical'
)

Found 20000 files belonging to 2 classes.


In [27]:
test_dataset= tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(150,150),
    batch_size=32,
    label_mode='categorical',
    shuffle=False
)

Found 5000 files belonging to 2 classes.


In [10]:
data_augmentation= Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
    layers.RandomTranslation(0.05,0.05)
])

In [11]:
def create_improved_model(l2_reg=0.001,dropout_rate=0.3):
  model= models.Sequential([
      data_augmentation,
      layers.Rescaling(1./255), #normalization

      #1st layer of cnn
      layers.Conv2D(32,(3,3),activation='relu',
                    kernel_regularizer = regularizers.l2(l2_reg),
                    input_shape=(150,150,3)),
      layers.BatchNormalization(),
      layers.MaxPooling2D((2,2)),
      layers.Dropout(dropout_rate*0.05),

      #2nd layer of CNN
      layers.Conv2D(64,(3,3),activation='relu',kernel_regularizer=regularizers.l2(l2_reg)),
      layers.BatchNormalization(),
      layers.MaxPooling2D((2,2)),
      layers.Dropout(dropout_rate*0.05),

      #3rd layer of cnn
      layers.Conv2D(64,(3,3),activation='relu',kernel_regularizer=regularizers.l2(l2_reg)),
      layers.BatchNormalization(),

      layers.Flatten(),
      #ann layer
      layers.Dense(256,activation='relu',kernel_regularizer=regularizers.l2(l2_reg)),
      layers.Dropout(dropout_rate),
      layers.Dense(2,activation='softmax')
  ])
  return model

In [12]:
model = create_improved_model(l2_reg=0.001,dropout_rate=0.3)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.compile(
    optimizer = optimizers.SGD(learning_rate=0.001, momentum=0.9),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [14]:
early_stopping = callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=10,
    restore_best_weights = True
)

In [15]:
reduce_lr= callbacks.ReduceLROnPlateau(
    moniter='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001
)

In [16]:
checkpoint = callbacks.ModelCheckpoint(
    'best_cat_vs_dog_model.keras',
    monitor='val_loss',
    save_best_only=True
)

In [19]:
history= model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs =5,
    callbacks=[early_stopping,reduce_lr,checkpoint]
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 44s 70ms/step - accuracy: 0.9516 - loss: 0.5617 - val_accuracy: 0.8482 - val_loss: 0.8575 - learning_rate: 2.5000e-04
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 50s 80ms/step - accuracy: 0.9526 - loss: 0.5520 - val_accuracy: 0.8668 - val_loss: 0.8152 - learning_rate: 2.5000e-04
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 77s 73ms/step - accuracy: 0.9559 - loss: 0.5477 - val_accuracy: 0.8600 - val_loss: 0.8347 - learning_rate: 2.5000e-04
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 45s 72ms/step - accuracy: 0.9561 - loss: 0.5439 - val_accuracy: 0.8598 - val_loss: 0.8544 - learning_rate: 2.5000e-04
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 83s 74ms/step - accuracy: 0.9589 - loss: 0.5356 - val_accuracy: 0.8726 - val_loss: 0.8066 - learning_rate: 2.5000e-04


In [28]:
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Final Test Loss: {test_loss:.2f}")
print(f"Final Test Accuracy: {test_acc:.2f}")

157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.8482 - loss: 0.8575
Final Test Loss: 0.86
Final Test Accuracy: 0.85


In [29]:
y_pred = model.predict(test_dataset)
y_pred_classes = np.argmax(y_pred, axis =1)

157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step


In [30]:
y_true_categorical = np.concatenate([y for x, y in test_dataset], axis=0)
y_true = np.argmax(y_true_categorical, axis=1)

In [31]:
print(classification_report(y_true,y_pred_classes,target_names=['Cat','Dog']))

              precision    recall  f1-score   support

         Cat       0.80      0.92      0.86      2500
         Dog       0.91      0.78      0.84      2500

    accuracy                           0.85      5000
   macro avg       0.86      0.85      0.85      5000
weighted avg       0.86      0.85      0.85      5000



In [32]:
print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_true, y_pred_classes))

=== CONFUSION MATRIX ===
[[2298  202]
 [ 557 1943]]


In [35]:
def predict_image(img_path):
  img = image.load_img(img_path,target_size=(150,150))
  img_array= image.img_to_array(img)
  img_array=np.expand_dims(img_array,axis=0)
  predictions = model.predict(img_array)

  class_names = ['Cat', 'Dog']
  predicted_class = class_names[np.argmax(predictions[0])]
  confidence = np.max(predictions[0])*100

  print(f"Image is : {predicted_class} with confidence level = {confidence:.2f}%")

In [36]:
predict_image('/content/petimage.jpg')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 328ms/step
Image is : Dog with confidence level = 96.98%


In [37]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 148, 148, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 72, 72, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 34, 34, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 73984)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    18,940,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           514 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,994,950 (144.94 MB)

 Trainable params: 18,997,314 (72.47 MB)

 Non-trainable params: 320 (1.25 KB)

 Optimizer params: 18,997,316 (72.47 MB)